In [1]:
import os
from pathlib import Path
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import numpy as np

from pygei.seges import alunos

In [2]:
saveFolder = Path (r'N:\05 - Arquivos Pessoais\Vinicius\painel_alertas_otimizacoes\dados')

NUM_ANO_LETIVO = '2026'

available_dates = [date for date in alunos.available_dates() if date.year == int(NUM_ANO_LETIVO)]

In [3]:
colunas_alvo = [
    'id_aluno', 'nm_aluno', 'data_nascimento', 'inep_escola', 
    'nm_escola', 'nm_regional', 'id_ano_letivo', 'num_ano_letivo', 'dt_matricula', 
    'dt_enturmacao', 'situacao_enturmacao', 'situacao_matricula', 'tipo_atendimento',
    'fl_deficiencia', 'dc_deficiencia', 'cpf', 'dc_cor_raca', "nome_ano_escolaridade",
    "ano_escolaridade", "nome_turma", "dc_turno", 'data_referencia'
]

dtype_dict = {
    'id_aluno': 'Int64', 
    'inep_escola': 'Int64', 
    'id_ano_letivo': 'Int64'
}

acumulado_base = []

for date in available_dates:
    df = alunos.load(date.year, date.month, date.day)

    df = df.astype(dtype_dict).copy()

    df.cpf = df.cpf.str.strip().str.replace(".", "").str.replace("-", "").str.zfill(11)

    df = df.seges.ativas()

    df.num_ano_letivo = df.num_ano_letivo.astype(str)

    df = df.seges.por_ano(NUM_ANO_LETIVO)

    numero_ano_letivo = [ano_letivo for ano_letivo in df.num_ano_letivo.unique() if 'MAPES' not in ano_letivo or 'CEET' not in ano_letivo]

    df = df[df.num_ano_letivo.isin(numero_ano_letivo)].copy()

    df['data_referencia'] = date

    df = df[colunas_alvo]

    acumulado_base.append(df)

C:\Users\alpaula\AppData\Local\Temp\ipykernel_23352\947421430.py:26: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.num_ano_letivo = df.num_ano_letivo.astype(str)
C:\Users\alpaula\AppData\Local\Temp\ipykernel_23352\947421430.py:26: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.num_ano_letivo = df.num_ano_letivo.astype(str)
C:\Users\alpaula\AppData\Local\Temp\ipykernel_23352\947421430.py:26: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_in

In [4]:
dfmatricula = pd.concat(acumulado_base)

In [5]:
table = pa.Table.from_pandas(dfmatricula)

saveFile = os.path.join(saveFolder, f'EMPILHADO_MATRICULAS.parquet')

pq.write_table(table, saveFile)